In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.3 MB/s eta 0:00:00


In [5]:
# Required Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    precision_score, accuracy_score, make_scorer
)
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# Add Gaussian noise to the data
def add_noise(data, noise_level=0.5):
    noise = np.random.normal(0, noise_level, data.shape)
    return data + noise

# Introduce missing values to simulate imperfect data
def add_missing_values(data, missing_rate=0.1):
    mask = np.random.rand(*data.shape) < missing_rate
    data[mask] = np.nan
    return data

# Load datasets with appropriate delimiter
stw_leg = pd.read_csv('/content/Stw_PToP_Leg.dat', delimiter="\\s+", header=None)
line_6s = pd.read_csv('/content/Line_6s_Th.dat', delimiter="\\s+", header=None)

# Name the columns for clarity
stw_leg.columns = ['Time', 'Leg_1', 'Leg_2', 'Leg_3', 'Leg_4', 'Leg_5', 'Leg_6']
line_6s.columns = ['Time', 'X', 'Y', 'Z', 'Roll', 'Pitch', 'Yaw']

# Remove the 'Time' column
stw_leg = stw_leg.drop(columns=['Time'])
line_6s = line_6s.drop(columns=['Time'])

# Add noise and missing values to features and labels
noise_level = 0.7
missing_rate = 0.3
X = add_missing_values(add_noise(stw_leg.values, noise_level=noise_level), missing_rate=missing_rate)
y = add_missing_values(add_noise(line_6s.values, noise_level=noise_level), missing_rate=missing_rate)

# Impute missing values with mean
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)
y = imputer.fit_transform(y)

# Normalize features and labels
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# Split dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Models to evaluate
models = {
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=10),
    "Decision Tree": DecisionTreeRegressor(max_depth=3, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=10, max_depth=3,
        min_samples_split=15, min_samples_leaf=10, random_state=42
    ),
    "Support Vector Machine (SVM)": SVR(kernel='rbf', C=1.0, epsilon=0.2),
    "Gradient Boosting Regressor": GradientBoostingRegressor(
        n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42
    )
}

# Function to calculate metrics
def evaluate_metrics(model, X, y, output_idx):
    y_true = y[:, output_idx]
    y_pred = model.predict(X)

    # Regression metrics
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Classification metrics
    y_true_bin = (y_true > 0).astype(int)
    y_pred_bin = (y_pred > 0).astype(int)
    accuracy = accuracy_score(y_true_bin, y_pred_bin)
    precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)

    return mse, mae, r2, accuracy, precision

# Initialize storage for results
results = []
overall_results = {}

# Train and evaluate models for each leg and output label
for model_name, model in models.items():
    model_results = [f"\n### {model_name} Results ###"]
    overall_metrics = {'mse': [], 'mae': [], 'r2': [], 'accuracy': [], 'precision': []}

    for leg_idx in range(1, 7):  # Legs 1 to 6
        for output_idx in range(y.shape[1]):  # Loop over each output label
            model.fit(X_train, y_train[:, output_idx])
            mse, mae, r2, accuracy, precision = evaluate_metrics(model, X_test, y_test, output_idx)

            # Aggregate metrics
            overall_metrics['mse'].append(mse)
            overall_metrics['mae'].append(mae)
            overall_metrics['r2'].append(r2)
            overall_metrics['accuracy'].append(accuracy)
            overall_metrics['precision'].append(precision)

    # Calculate overall averages
    overall_results[model_name] = {
        'MSE': np.mean(overall_metrics['mse']),
        'MAE': np.mean(overall_metrics['mae']),
        'R²': np.mean(overall_metrics['r2']),
        'Accuracy': np.mean(overall_metrics['accuracy']),
        'Precision': np.mean(overall_metrics['precision']),
    }

# Print overall results
print("### Overall Results ###")
for model_name, metrics in overall_results.items():
    print(f"{model_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")


### Overall Results ###
K-Nearest Neighbors:
  MSE: 0.4116
  MAE: 0.5226
  R²: 0.6194
  Accuracy: 0.8226
  Precision: 0.7520
Decision Tree:
  MSE: 0.6271
  MAE: 0.5945
  R²: 0.4116
  Accuracy: 0.7796
  Precision: 0.7150
Random Forest:
  MSE: 0.4679
  MAE: 0.5352
  R²: 0.5662
  Accuracy: 0.8280
  Precision: 0.7684
Support Vector Machine (SVM):
  MSE: 0.5131
  MAE: 0.5578
  R²: 0.5214
  Accuracy: 0.8387
  Precision: 0.7813
Gradient Boosting Regressor:
  MSE: 0.5277
  MAE: 0.5340
  R²: 0.5061
  Accuracy: 0.8226
  Precision: 0.7630
